In [38]:
import cvxpy as cp
import numpy as np
import pandas as pd
import os

In [39]:
os.getcwd()

'c:\\Users\\thewo\\Documents\\Git\\OptimalChargerPlacement'

In [40]:
growth = pd.read_csv('OptimizationInput\\ev_load_growth_from2025unmanaged_to2035managed_slice_hour_20.csv')
conMatrix = pd.read_csv('OptimizationInput\\feederBlockIntersectionMtx.csv')

add_load = growth['load'].to_numpy()
mat = conMatrix.to_numpy()
demand = mat@add_load

cap_max = pd.read_csv('OptimizationInput\\alamedaMinICA_slice_month6_hour20.csv')
cap_max['IC_min_kW'] = cap_max['IC_min_kW'].fillna(0)
maxes = cap_max['IC_min_kW'].to_numpy()

In [62]:
n = 303
Cm = np.random.rand(n,n)*25
np.fill_diagonal(Cm, 0)
Cm = np.square(Cm)

In [69]:

ni = cp.Variable(n, boolean=True)
x = cp.Variable(n)
y = cp.Variable((n,n))

Cu = 0.025
Cf = 4

# Replace with distance matrix
M=10**6
dem = 20*demand


In [70]:
obj = Cf*cp.sum(ni) + Cu*cp.sum(x) + cp.sum(cp.multiply(Cm, y))

constraints = []
constraints += [x <= ni*M, x >= 0, y >= 0]

for i in range(n):
    constraints += [x[i]+maxes[i] >= cp.sum(y[:,i])-cp.sum(y[i,:])+y[i,i]]
    constraints += [cp.sum(y[i,:]) >= dem[i]]

prob = cp.Problem(cp.Minimize(obj), constraints)


In [71]:
prob.solve(verbose=True)

(CVXPY) Dec 08 01:16:19 AM: Your problem has 92415 variables, 93021 constraints, and 0 parameters.
(CVXPY) Dec 08 01:16:19 AM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Dec 08 01:16:19 AM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Dec 08 01:16:19 AM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Dec 08 01:16:19 AM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Dec 08 01:16:19 AM: Compiling problem (target solver=SCIPY).
(CVXPY) Dec 08 01:16:19 AM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ConeMatrixStuffing -> SCIPY
(CVXPY) Dec 08 01:16:19 AM: Applying reduction Dcp2Cone
(CVXPY) Dec 08 01:16:19 AM: Applying reduction CvxAttr2Constr
(CVXPY) Dec 08 01:16:19 AM: Applying reduction ConeMatrixStuffing


                                     CVXPY                                     
                                     v1.7.3                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------


(CVXPY) Dec 08 01:16:23 AM: Applying reduction SCIPY
(CVXPY) Dec 08 01:16:23 AM: Finished problem compilation (took 4.143e+00 seconds).
(CVXPY) Dec 08 01:16:23 AM: Invoking solver SCIPY  to obtain a solution.


-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------


(CVXPY) Dec 08 01:18:04 AM: Problem status: optimal
(CVXPY) Dec 08 01:18:04 AM: Optimal value: 3.032e+04
(CVXPY) Dec 08 01:18:04 AM: Compilation took 4.143e+00 seconds
(CVXPY) Dec 08 01:18:04 AM: Solver (including time spent in interface) took 1.010e+02 seconds


Solver terminated with message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
-------------------------------------------------------------------------------
                                    Summary                                    
-------------------------------------------------------------------------------


np.float64(30317.759928624422)

In [72]:
np.sum(ni.value >= 0.1)

np.int64(45)

In [67]:
x.value

array([   -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,    -0.        ,    -0.        ,    -0.        ,
          -0.        ,  1607.59047829,    -0.        ,    -0.        ,
          -0.        ,    -0.        , 14641.78518318,    -0.        ,
      

In [68]:
y.value

array([[-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.],
       ...,
       [-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.],
       [-0., -0., -0., ..., -0., -0., -0.]], shape=(303, 303))

In [48]:
growth.columns

Index(['geoid', 'geoid_str', 'load'], dtype='object')

In [49]:
np.sum(y.value[1,:])

np.float64(2028.4336132882486)

In [50]:
demand

array([3.88316848e+01, 1.01421681e+02, 5.73582451e+01, 7.48227969e+01,
       3.62526029e+01, 8.87732264e+01, 1.89056942e+02, 6.67578838e+02,
       4.03278168e+01, 2.45097242e+02, 3.48055382e+02, 3.60334948e+02,
       3.22176206e+02, 4.06476365e+02, 1.61389590e+02, 8.35210625e+01,
       1.16398763e+01, 6.02493148e+01, 1.64630856e+02, 1.76618950e+01,
       2.16441463e+02, 6.28534641e+01, 5.71736776e+01, 5.60600639e+01,
       7.15250510e+01, 9.06410984e+01, 5.14983953e+01, 8.23835198e+01,
       8.34626795e+01, 1.81552904e+02, 6.68104450e+02, 1.41648989e+02,
       1.10356311e+02, 4.92528206e+02, 1.61517107e+02, 4.12442660e+02,
       2.61895959e+02, 2.87384374e+02, 8.87976663e+01, 3.24359070e+02,
       4.09775881e+02, 2.58096242e+02, 1.36707534e+02, 6.36783827e+02,
       4.25993651e+01, 1.49499352e+02, 1.20745268e+02, 1.26996507e+02,
       3.48364929e+01, 8.03795239e+01, 4.11227169e+01, 1.41194240e+01,
       1.71998361e+02, 1.36330299e+02, 1.48181820e+02, 2.33593988e+02,
      

In [51]:
cap_max['IC_min_kW'] = cap_max['IC_min_kW'].fillna(0)

In [52]:
cap_max

,feeder_id,IC_min_kW
0,12011101,0.0000
1,12011102,0.0000
2,12011103,0.0000
3,12011104,0.0000
4,12011105,0.0000
...,...,...
298,14722111,4084.2234
299,82831109,1269.2465
300,163741102,224.9415
301,163741103,357.6861


In [53]:
maxes = cap_max['IC_min_kW'].to_numpy()

In [54]:
demand > maxes

array([ True,  True,  True,  True,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False,  True, False, False, False, False,
       False, False,  True, False, False, False, False, False,  True,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False,  True, False, False, False, False,
       False, False, False, False,  True,  True,  True,  True,  True,
       False, False, False, False, False,  True,  True, False, False,
       False, False, False,  True,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True,  True,  True,  True,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False,  True,
        True, False, False,  True, False, False, False, False,  True,
       False, False,

In [55]:
np.sum(demand)-np.sum(maxes)

np.float64(-467249.564158474)

In [56]:
np.sum(demand)

np.float64(159087.78614102595)